In [36]:
import pickle
import numpy as np
import os
import pandas as pd
import math
import json
from tqdm import tqdm
from embedding import BertHuggingface
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

from itertools import chain
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.utils import shuffle

import torch
import transformers

sns.set(font_scale=2)

In [37]:
REVIEW_DIR = "../../data/biosbias_review_artifacts/review_v2_fixed_indices/"
TITLE_JSON = "data/title_lookup.json"

reviewed_classes = ['architect', 'surgeon', 'dentist', 'teacher', 'psychologist', 'nurse', 'photographer', 'physician', 'attorney', 'journalist', 'software_engineer', 'model']

COLUMNS_TO_MERGE = ['index', 'auto_raw_title', 'auto_title', 'review', 'raw_titles', 'titles', 'valid', 'style_valid', 'comment']
COLUMNS_TO_KEEP = ['index', 'path', 'gender', 'start_pos', 'URI', 'auto_raw_title', 'auto_title', 'review', 'raw_titles', 'titles', 'valid', 'style_valid']
SAVE_DIR_REVIEW = "data/BIOS_REVIEWED_v2.csv" # the reviewed artifacts merged into one file, including biographies
SAVE_DIR_MERGED = "data/BIOS_MERGED_v2.csv" # review labels merged into the full dataset
SAVE_DIR_PATCH = "data/BIOS_LABELS_v2.csv" # patch for the full dataset, biographies are removed

BIOS_RAW = "../../data/biosbias2023/BIOS.pkl"

SEP = '\t'

In [38]:
with open(TITLE_JSON, 'r') as j:
    raw_titles_per_title = json.load(j)
    
classes = [key for key in raw_titles_per_title.keys() if not '-' in key]
raw_titles = [elem for val in raw_titles_per_title.values() for elem in val]
print(classes)

['architect', 'attorney', 'dentist', 'journalist', 'nurse', 'photographer', 'physician', 'psychologist', 'surgeon', 'teacher', 'model', 'software_engineer', 'professor', 'accountant', 'chiropractor', 'personal trainer', 'filmmaker', 'dietitian', 'dj', 'composer', 'poet', 'comedian', 'yoga teacher', 'interior designer', 'pastor', 'rapper', 'paralegal', 'painter', '*artist/ designer', '*engineer', '*actor', '*writer', '*trainer', '*consultant/coaches', '*director/producer', '*editor', '*researcher', '*entrepeneur']


## 1) Merge csv files from review

### Merge dataframes of individual classes and rename columns

In [39]:
dfs = []
for cls in reviewed_classes:
    print(cls)
    dfs.append(pd.read_csv(REVIEW_DIR+cls+'.csv', sep=SEP))
df = pd.concat(dfs)
df = df.rename(columns={'label': 'raw_titles', 'raw_title': 'auto_raw_title', 'title': 'auto_title', 'stylish valid': 'style_valid'})
df['index'] = df['index'].astype(int)
df = df.reset_index(drop=True)

architect
surgeon
dentist
teacher
psychologist
nurse
photographer
physician
attorney
journalist
software_engineer
model


### Check if all raw titles are covered by the json lookup file

In [40]:
# check for raw titles not in the json
unique_entries = set(df[~df['raw_titles'].isna()]['raw_titles'])
unique_entries = [entry.rstrip().lstrip().lower() for entry in unique_entries]
entry_list = ', '.join(unique_entries).split(', ')
new_raw_titles = list(set(entry_list))
new_raw_titles = [entry.rstrip() for entry in new_raw_titles] # for some reason there are still spaces on the right side

for title in new_raw_titles:
    if title not in raw_titles:
        print('\"'+title+'\"')

# the 44 % one is in the lookup but matching doesnt work

"44 %"


### Map raw titles to tiles

In [41]:
# Create inverse mapping
inverse_map = {}
for canonical, raws in raw_titles_per_title.items():
    for raw in raws:
        inverse_map[raw] = canonical

# Apply mapping
def map_titles(raw_str):
    if pd.isna(raw_str):
        return []
    raws = [r.strip() for r in str(raw_str).split(', ') if r.strip()]
    return [inverse_map.get(r, r) for r in raws]  # fallback to raw if not found

def to_list(raw_str):
    if pd.isna(raw_str):
        return []
    return [r.strip() for r in str(raw_str).split(', ') if r.strip()]

df['titles'] = df['raw_titles'].apply(map_titles)
df['raw_titles'] = df['raw_titles'].apply(to_list)

### Save reviewed data

In [42]:
# reviewed data as one csv
df['review'] = 1 # some were automatically set to > 1 when filtering data and not manually changed to 1
df.to_csv(SAVE_DIR_REVIEW, sep=SEP, index=False)

## 2) Map to full dataset

In [43]:
# remove entries that could not be mapped back (index -1)
df = df[df['index'] >= 0]

In [44]:
# merge with full data
with open(BIOS_RAW, 'rb') as handle:
    full_data = pickle.load(handle)
df_full = pd.DataFrame(full_data)
#df_full = df_full.rename(columns={'raw_title': 'auto_raw_title', 'title': 'auto_title'})
df_full['index'] = df_full.index
df_full

,path,raw,name,raw_title,gender,start_pos,title,URI,bio,index
0,crawl-data/CC-MAIN-2013-20/segments/1368696381...,* Nora Fisher Onar is an assistant professor o...,"(Nora, Fisher, Onar)",assistant professor,F,109,professor,http://acturca.wordpress.com/2012/04/13/turkey...,_ is also a Ronald D. Asmus Policy Entrepreneu...,0
1,crawl-data/CC-MAIN-2013-20/segments/1368696381...,Brent Wagner is a certified public accountant ...,"(Brent, , Wagner)",certified public accountant,M,124,accountant,http://www.forwardtheater.com/about/board,_ is a member of the AICPA and WICPA. _ gradua...,1
2,crawl-data/CC-MAIN-2014-41/segments/1410657102...,Dr. Shawn Zelig Aster is an assistant professo...,"(Shawn, Zelig, Aster)",assistant professor,M,79,professor,http://yuspeakers.org/speakers/,Dr. _ has held teaching and research positions...,2
3,crawl-data/CC-MAIN-2013-20/segments/1368696381...,Brittany McCandless is a journalist with a pri...,"(Brittany, , McCandless)",journalist,F,74,journalist,http://morevoices.tumblr.com/,_ currently works on CNN’s newest primetime sh...,3
4,crawl-data/CC-MAIN-2013-20/segments/1368696381...,Arturo Toledo is an Architect. He runs a bouti...,"(Arturo, , Toledo)",Architect,M,30,architect,http://diggthedrazen.com/author/diggthedrazen/,_ runs a boutique design studio attending clie...,4
...,...,...,...,...,...,...,...,...,...,...
397902,crawl-data/CC-MAIN-2018-43/segments/1539583519...,Rebecca Wexler is a documentary filmmaker; co-...,"(Rebecca, , Wexler)",filmmaker,F,322,filmmaker,https://www.thecommonsjournal.org/articles/10....,_ recently returned from working as a Fulbrigh...,397902
397903,crawl-data/CC-MAIN-2018-43/segments/1539583519...,Fernando Quesada is an Attorney at-law admitte...,"(Fernando, , Quesada)",Attorney,M,66,attorney,https://dentons.lopez-velarde.com/en/fernando-...,_ focuses _ practice in all forms of Energy (i...,397903
397904,crawl-data/CC-MAIN-2018-43/segments/1539583519...,Jeffrey Kucik is an assistant professor in the...,"(Jeffrey, , Kucik)",assistant professor,M,115,professor,http://www.latimes.com/opinion/op-ed/la-oe-kuc...,_ maintains the trade policy site www.trademon...,397904
397905,crawl-data/CC-MAIN-2018-43/segments/1539583519...,Leslie E. Seawright is an Assistant Professor ...,"(Leslie, E, Seawright)",Assistant Professor,F,107,professor,https://www.abebooks.com/9781443863339/Going-G...,"_ teaches courses in Technical Writing, Compos...",397905


In [ ]:
# Merge using left_index for df, right_index or right on index for df_full
merged_df = pd.merge(
    df_full,
    df[COLUMNS_TO_MERGE],
    on='index',
    how='left'
)
merged_df

In [46]:
# set default values for samples not part of review
merged_df.loc[merged_df['review'] != 1, 'review'] = 0
merged_df.loc[merged_df['review'] != 1, 'valid'] = -1
merged_df.loc[merged_df['review'] != 1, 'style_valid'] = -1
merged_df['auto_raw_title'] = merged_df['raw_title']
merged_df['auto_title'] = merged_df['title']

### Save the full merged dataset and the label patch without sensitive information

In [47]:
# save full dataset with review
merged_df.to_csv(SAVE_DIR_MERGED, sep=SEP, index=False)

# save patch
print(SAVE_DIR_PATCH)
print(COLUMNS_TO_KEEP)
merged_df[COLUMNS_TO_KEEP].to_csv(SAVE_DIR_PATCH, sep=SEP, index=False)

data/BIOS_LABELS_v2.csv
['index', 'path', 'gender', 'start_pos', 'URI', 'auto_raw_title', 'auto_title', 'review', 'raw_titles', 'titles', 'valid', 'style_valid']


## 3) Create mapping to huggingface dataset


In [48]:
# load huggingface dataset
from datasets import load_dataset
dataset = load_dataset("LabHC/bias_in_bios")

# labels in the order specified here: https://huggingface.co/datasets/LabHC/bias_in_bios
# not included in the datasets metadata
huggingface_label = ['accountant', 'architect', 'attorney', 'chiropractor', 'comedian', 'composer', 'dentist', 'dietitian', 'dj', 'filmmaker', 'interior_designer', 'journalist', 'model', 'nurse', 'painter', 'paralegal', 'pastor', 'personal_trainer', 'photographer', 'physician', 'poet', 'professor', 'psychologist', 'rapper', 'software_engineer', 'surgeon', 'teacher', 'yoga_teacher']
partitions = ['train', 'test', 'dev']

In [49]:
labeled_data = merged_df.to_dict('records')

for sample in labeled_data:
    sample['match_text'] = sample['raw'][sample['start_pos']:].strip()

### Create initial mapping for both directions

In [50]:
MATCH_PKL = 'data/matches_v2.pkl'
if not os.path.isfile(MATCH_PKL):
    matches = {part: [] for part in partitions}
    reverse_matches = [[] for sample in labeled_data]

    for part in partitions:
        matches[part] = []
        print(part)
        for i in tqdm(range(len(dataset[part]))):
            text = dataset[part][i]['hard_text']
            f = list(filter(lambda labeled_data: labeled_data['match_text'] == text, labeled_data))
            potential_matches = []
            for elem in f:
                if elem['auto_title'] == huggingface_label[dataset[part][i]['profession']]:
                    potential_matches.append(elem['index'])
                    reverse_matches[elem['index']].append((part,i))
            matches[part].append((i,potential_matches))

    with open(MATCH_PKL, 'wb') as handle:
        pickle.dump({'matches': matches, 'reverse_matches': reverse_matches}, handle)
    
else:
    print("load precomputed matches")
    with open(MATCH_PKL, 'rb') as handle:
        loaded_matches = pickle.load(handle)
        matches = loaded_matches['matches']
        reverse_matches = loaded_matches['reverse_matches']   

load precomputed matches


### Summarize match success

- the huggingface dataset is based on another crawl of the dataset, which may not perfectly overlap with our crawled version (crawler skipping in-accessible sources)
- the dataset contains duplicates where the same person appears in multiple biography samples, some of these biographies are (almost) identical (e.g. duplicate starts with "Description: " followed by the exact same biography, i.e. hard\_text (second sentence onward is identical)

In [51]:
for part in partitions:
    print(part)
    count0 = 0
    count_double = 0

    for match in matches[part]:
        if match[1] == []:
            count0 += 1
        elif len(match[1]) > 1:
            count_double += 1

    print("got %i samples in total" % len(matches[part]))
    print("for %i samples found no match in the labeled data" % count0)
    print("for %i samples found multiple matches in the labeled data" % count_double)
    

count0 = 0
count_multi = 0
for match in reverse_matches:
    n_matches = len(match)
    if n_matches == 0:
        count0 += 1
    if n_matches > 1:
        count_multi += 1
        
print("based on the crawled dataset")
print("got %i samples in total" % len(reverse_matches))
print("for %i samples found no match in the labeled data" % count0)
print("for %i samples found multiple matches in the labeled data" % count_multi)

train
got 257478 samples in total
for 1197 samples found no match in the labeled data
for 933 samples found multiple matches in the labeled data
test
got 99069 samples in total
for 456 samples found no match in the labeled data
for 343 samples found multiple matches in the labeled data
dev
got 39642 samples in total
for 171 samples found no match in the labeled data
for 145 samples found multiple matches in the labeled data
based on the crawled dataset
got 397907 samples in total
for 3530 samples found no match in the labeled data
for 1412 samples found multiple matches in the labeled data


### Unify mapping and merge datasets

Handle duplicates in mapping.

In [ ]:
dataset_for_filter = []
for part in partitions:
    for i, sample in enumerate(dataset[part]):
        sample_for_filter = sample.copy()
        sample_for_filter.update({'split': part, 'id': i})
        dataset_for_filter.append(sample_for_filter)
        
dataset_for_filter[0]

In [53]:
merged_dataset = {part: [] for part in partitions}
dup_id_list = []
asserts = []

for part in partitions:
    print(part)
    for match in tqdm(matches[part]):
        # match[0] is the huggingface id (of part); match[1] is a list of ids in the review patch
        idx = match[0]
        sample = dataset[part][idx]
        text = sample['hard_text']

        # add ids of duplicates to each entry
        f = list(filter(lambda dataset_for_filter: dataset_for_filter['hard_text'] == text, dataset_for_filter))
        sample.update({'text_duplicates': [(elem['split'], elem['id']) for elem in f if (elem['id'] != idx or elem['split'] != part)]})

        if len(match[1]) == 0:
            # no match in the reviewed data: keep old label
            sample.update({'titles_supervised': [], 'raw_titles_supervised': [], 'gender_supervised': '', 'auto_title': '', 'auto_raw_title': '', 'path': '', 'URI': '', 
                           'review': 0, 'valid': -1, 'style_valid': -1, 'raw': '', 'bio': '', 'name': ('',''), 'duplicates': []})
            merged_dataset[part].append(sample)
        elif len(match[1]) > 1:
            # multiple matches with review data/ duplicates exist

            # remember this is a sample with multiple matches (duplicates) 
            dup_id_list.append(match[0])

            # collect ids of duplicates in the hugginface dataset via reverse matches (review -> huggingface)
            duplicates = []
            for m_id in match[1]:
                for d_tup in reverse_matches[m_id]:
                    if d_tup[1] != idx and d_tup not in duplicates:
                        duplicates.append(d_tup)

            # check if duplicates have the same title and gender assigned; in that case it doesn't matter which is taken for the merge
            same_number_of_duplicates = (len(duplicates)+1 == len(match[1]))
            all_equal = True
            if not same_number_of_duplicates:
                base_sample = labeled_data[match[1][0]]
                #if len(duplicates) > 0 and len(match[1]) > 1:
                #    print("inconsistent number of duplicates: %i vs. %i" % (len(duplicates)+1, len(match[1])))
                #    print(idx, match[1][0], "base sample: ")
                #    print_sample(base_sample)
                for next_id in match[1][1:]:
                    comp_sample = labeled_data[next_id]
                    if not (comp_sample['titles'] == base_sample['titles'] and comp_sample['gender'] == base_sample['gender']):
                        all_equal = False
                if not all_equal:
                    asserts.append(idx)
#                assert all_equal, "inconsistent number of duplicates in both dataset versions"+str(match[1])+ " vs. " + str(duplicates)+" for idx "+str(idx)

                # take first possible match
                match_id = match[1][0]
                merge_sample = labeled_data[match_id]

                # assign other match ids to duplicates (1:1), handle inconsistent numbers of duplicates!
                for i, (d_split,d_idx) in enumerate(duplicates):
                    if i+1 >= len(matches[d_split][d_idx][1]):
                        i -= len(matches[d_split][d_idx][1])
                    matches[d_split][d_idx] = (matches[d_split][d_idx][0], [matches[d_split][d_idx][1][i+1]])
                    #print(d_split, d_idx, matches[d_split][d_idx][1], "->", [matches[d_split][d_idx][1][i]])
            else:
                # take first possible match
                match_id = match[1][0]
                merge_sample = labeled_data[match_id]

                # assign other match ids to duplicates (1:1)
                for i, (d_split,d_idx) in enumerate(duplicates):
                    index = min(i+1,len(matches[d_split][d_idx][1])-1)
                    matches[d_split][d_idx] = (matches[d_split][d_idx][0], [matches[d_split][d_idx][1][index]])
                    #print(matches[d_split][d_idx][1], "->", [matches[d_split][d_idx][1][i]])

            sample.update({'titles_supervised': merge_sample['titles'], 'raw_titles_supervised': merge_sample['raw_titles'], 'gender_supervised': merge_sample['gender'], 
                           'auto_title': merge_sample['auto_title'], 'auto_raw_title': merge_sample['auto_raw_title'],  'path': merge_sample['path'], 'URI': merge_sample['URI'], 
                           'review': merge_sample['review'], 'valid': merge_sample['valid'], 'style_valid': merge_sample['style_valid'], 'raw': merge_sample['raw'], 
                           'bio': merge_sample['bio'], 'name': merge_sample['name'], 'duplicates': duplicates})
            merged_dataset[part].append(sample)

        else:
            # exactly one match
            match_id = match[1][0]
            merge_sample = labeled_data[match_id]
            sample.update({'titles_supervised': merge_sample['titles'], 'raw_titles_supervised': merge_sample['raw_titles'], 'gender_supervised': merge_sample['gender'], 
                           'auto_title': merge_sample['auto_title'], 'auto_raw_title': merge_sample['auto_raw_title'],  'path': merge_sample['path'], 'URI': merge_sample['URI'], 
                           'review': merge_sample['review'], 'valid': merge_sample['valid'], 'style_valid': merge_sample['style_valid'], 'raw': merge_sample['raw'], 
                           'bio': merge_sample['bio'], 'name': merge_sample['name'], 'duplicates': []})
            merged_dataset[part].append(sample)

print(f"found {len(asserts)} samples that would have triggered the all equal assert")

train


100%|████████████████| 257478/257478 [3:13:24<00:00, 22.19it/s]


test


100%|██████████████████| 99069/99069 [1:12:24<00:00, 22.80it/s]


dev


100%|████████████████████| 39642/39642 [28:53<00:00, 22.87it/s]

found 13 samples that would have triggered the all equal assert


In [55]:
# make sure duplicates correctly cross-reference each other
for i, sample in enumerate(merged_dataset[part]):
    for (split, idx) in sample['duplicates']:
        if len(merged_dataset[split][idx]['duplicates']) == 0:
            dup = [tup for tup in sample['duplicates'] if not tup == (split,idx)]
            dup.append((part, i))
            merged_dataset[split][idx]['duplicates'] = dup

### Save merged data (not published)

In [56]:
with open('data/huggingface_merge_complete_v2.pkl', 'wb') as handle:
    pickle.dump(merged_dataset, handle)

### Save public patch with labels and references to huggingface dataset

In [57]:
public_data = {part: [] for part in partitions}
keys_to_copy = ['text_duplicates', 'titles_supervised', 'raw_titles_supervised', 'gender_supervised', 'auto_title', 'auto_raw_title', 'review', 'valid', 'style_valid', 'duplicates']

for part in partitions:
    for entry in merged_dataset[part]:
        reduced_entry = {k: entry[k] for k in keys_to_copy}
        public_data[part].append(reduced_entry)
        
with open('data/huggingface_patch_v2.pkl', 'wb') as handle:
    pickle.dump(public_data, handle)

### Inspect duplicates (test-set contamination)

In [58]:
test = 'test'
contaminations = []
count_dup = 0
for part in partitions:
    for i, entry in enumerate(merged_dataset[part]):
        for (split, idx) in entry['duplicates']:
            count_dup += 1
            if split != part:
                if part == test:
                    contaminations.append((i, split, idx))

print(f"found {count_dup} duplicates across all splits")
print(f"found {len(contaminations)} samples in the test set with duplicates in the train or dev split!")

found 782 duplicates across all splits
found 30 samples in the test set with duplicates in the train or dev split!


In [61]:
per_job = {}
for (i, split, idx) in contaminations:
    print(split, idx, merged_dataset[split][idx]['auto_title'], merged_dataset[split][idx]['duplicates'])
    print('test', i, merged_dataset['test'][i]['auto_title'], merged_dataset['test'][i]['duplicates'])
    print()
    job = merged_dataset['test'][i]['auto_title']
    if job not in per_job.keys():
        per_job[job] = 1
    else:
        per_job[job] += 1

print(per_job)

dev 14882 model []
test 5312 model [('dev', 14882)]

dev 17122 professor []
test 7576 professor [('dev', 17122)]

dev 17153 software_engineer []
test 10686 software_engineer [('dev', 17153)]

dev 6532 dentist []
test 11309 dentist [('dev', 6532)]

dev 17382 chiropractor []
test 11974 chiropractor [('dev', 17382)]

dev 34610 attorney []
test 14250 attorney [('dev', 34610)]

dev 32059 poet []
test 14262 poet [('dev', 32059)]

dev 38847 professor []
test 17501 professor [('dev', 38847)]

dev 3033 comedian []
test 21149 comedian [('dev', 3033)]

dev 17268 attorney []
test 21502 attorney [('dev', 17268)]

dev 4794 professor []
test 25731 professor [('dev', 4794)]

dev 25245 professor []
test 27739 professor [('dev', 25245)]

dev 20167 physician []
test 29140 physician [('dev', 20167)]

dev 33041 software_engineer []
test 38458 software_engineer [('dev', 33041)]

dev 23459 professor []
test 40766 professor [('dev', 23459)]

dev 27755 dietitian []
test 40936 dietitian [('dev', 27755)]

dev 32